In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor

    transforms.ToTensor(),

    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])
# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define mean & std for denormalization (EfficientNet Preprocessing)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [270,233,110,89,15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image & label

    # Convert tensor to numpy for visualization
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)

    # Denormalize the image
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].axis('off')

plt.show()

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Write your code here
device = "cuda" if torch.cuda.is_available() else "cpu"
efficientnet = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
efficientnet.eval().to(device)

In [ ]:
#Freeze the backbone
for param in efficientnet.features.parameters():
    param.requires_grad = False

# Ensure num_classes is corect for EMNIST letters
num_classes = 26

# Replace the classifier head
# We need to match the output features to the number of classes (26).
in_features = efficientnet.classifier[1].in_features
efficientnet.classifier[1] = nn.Linear(in_features, num_classes)

# Move the model to the correct device
efficientnet = efficientnet.to(device)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        # adjust labels
        labels = labels - 1

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # Adjust labels
            labels = labels - 1

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / total
    val_acc = 100. * correct / total
    return val_loss, val_acc

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

# Create test loader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientnet.parameters(), lr=0.001)

# Lists to store metrics
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

epochs = 5

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train_loss, train_acc = train_one_epoch(efficientnet, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(efficientnet, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Plotting
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.show()

In [ ]:
def validate_tta(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # Adjust labels
            labels = labels - 1

            # 1. Original predictions
            outputs = model(images)

            # 2. Horizontally flipped predictions
            h_flipped = torch.flip(images, dims=[3])
            outputs_h = model(h_flipped)

            # 3. Vertically flipped predictions
            v_flipped = torch.flip(images, dims=[2])
            outputs_v = model(v_flipped)

            # Average the predictions
            avg_outputs = (outputs + outputs_h + outputs_v) / 3.0

            loss = criterion(avg_outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = avg_outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / total
    val_acc = 100. * correct / total
    return val_loss, val_acc

# Run TTA evaluation
print("Running Test Time Augmentation (TTA) Evaluation...")
tta_loss, tta_acc = validate_tta(efficientnet, test_loader, criterion, device)
print(f"TTA Test Loss: {tta_loss:.4f}, TTA Test Acc: {tta_acc:.2f}%")